# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
%pip -q install duckdb huggingface_hub scikit-learn
import os, getpass
HF_TOKEN = os.environ.get('HF_TOKEN') or getpass.getpass('Paste your Hugging Face READ token (hf_...): ')

import duckdb
con = duckdb.connect()
con.execute(f"CREATE OR REPLACE SECRET hf (TYPE huggingface, TOKEN '{HF_TOKEN}')")

MARCH_PATH = "hf://datasets/FlyRank/internship-warehouse/fact_content_daily_performance/month=2026-03/data_0.parquet"

Paste your Hugging Face READ token (hf_...): ··········


## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

Method choice: Random Forest classifier.

My lane is a ranking/prioritization problem (which pages to review first),
and Random Forest handles the mix of noisy, partially-missing, non-linear
signals (impressions, position, CTR gaps) better than a single linear rule
or a single decision tree — the earlier notebooks already showed no single
feature separates decliners well alone (AUC ~0.5 each). A forest can combine
several weak signals, which matches what my baseline rule was already trying
to do by hand with two flags. Random Forest also gives permutation
importance for free, which is exactly what Section 4 needs for interpreting
errors.

In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GroupShuffleSplit
from sklearn.metrics import classification_report

print("Method: RandomForestClassifier — handles noisy, non-linear signals; "
      "gives permutation importance for interpretation.")


Method: RandomForestClassifier — handles noisy, non-linear signals; gives permutation importance for interpretation.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

Split design: GroupShuffleSplit grouped by client_hash_id.

A random row-level split would let the same client's pages appear in both
train and test — the model could "memorize" a client's baseline traffic
level instead of learning a generalizable pattern. Grouping by
client_hash_id means an entire client's pages go to one side only, so the
test score reflects performance on clients the model has never seen —
the honest question for a capstone recommendation.

In [4]:
# Build the labeled dataset: features from first half of March, label = second half vs first half decline
data = con.sql(f"""
    WITH bounds AS (
        SELECT MIN(report_date) AS start_d, MAX(report_date) AS end_d
        FROM read_parquet('{MARCH_PATH}')
    ),
    windowed AS (
        SELECT client_hash_id, content_hash_id,
               SUM(CASE WHEN report_date > b.start_d + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_last15,
               SUM(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY THEN gsc_impressions ELSE 0 END) AS imp_prev15,
               AVG(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY THEN gsc_avg_position END) AS avg_position_prev,
               AVG(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY THEN gsc_clicks * 1.0 / NULLIF(gsc_impressions,0) END) AS ctr_prev,
               STDDEV(CASE WHEN report_date <= b.start_d + INTERVAL 15 DAY THEN gsc_impressions END) AS impression_volatility_prev
        FROM read_parquet('{MARCH_PATH}') f, bounds b
        WHERE gsc_data_available IS TRUE
        GROUP BY client_hash_id, content_hash_id
        HAVING imp_prev15 >= 50
    )
    SELECT *, (imp_last15 < 0.8 * imp_prev15)::INT AS is_declining
    FROM windowed
""").df()

print(f"{len(data):,} content items | decline rate: {data['is_declining'].mean():.3f}")

feature_cols = ['imp_prev15', 'avg_position_prev', 'ctr_prev', 'impression_volatility_prev']
data = data.dropna(subset=feature_cols)
print(f"after dropna: {len(data):,} rows")

gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(data, groups=data['client_hash_id']))
train, test = data.iloc[train_idx], data.iloc[test_idx]

print(f"train: {len(train):,} rows, {train['client_hash_id'].nunique()} clients")
print(f"test:  {len(test):,} rows, {test['client_hash_id'].nunique()} clients")
print("client overlap between train/test:",
      len(set(train['client_hash_id']) & set(test['client_hash_id'])), "(should be 0)")


94,559 content items | decline rate: 0.373
after dropna: 94,517 rows
train: 69,079 rows, 30 clients
test:  25,438 rows, 10 clients
client overlap between train/test: 0 (should be 0)


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

Comparing the trained model against my Week-4 baseline rule, on the same
held-out test clients (grouped split, no client overlap), same label
(15-day decline), same metric (Precision@50).

Result: Random Forest achieves Precision@50 = 0.66 vs the baseline rule's
0.24 — roughly 2.75x better at correctly flagging the top 50 declining
pages. This is a meaningful, honest improvement, not just added complexity:
the same features that individually showed weak signal (Task-4's AUC ~0.5)
combine into a real prioritization gain when the model learns interactions
between them.

In [5]:
import numpy as np
import pandas as pd

def precision_at_k(scores, labels, k):
    order = np.argsort(-np.asarray(scores))
    return np.asarray(labels)[order[:k]].mean()

X_train, y_train = train[feature_cols], train['is_declining']
X_test, y_test = test[feature_cols], test['is_declining']

model = RandomForestClassifier(n_estimators=300, max_depth=6, random_state=42, n_jobs=-1)
model.fit(X_train, y_train)
model_scores = model.predict_proba(X_test)[:, 1]

# Baseline rule from Task-5, recomputed on the SAME test rows (uses only prev-window features, no leakage)
test = test.copy()
test['position_tier'] = pd.cut(
    test['avg_position_prev'], bins=[0, 3, 10, 20, 1000], labels=['1-3', '4-10', '11-20', '20+']
)
tier_avg_ctr = test.groupby('position_tier')['ctr_prev'].transform('mean')
test['ctr_gap'] = tier_avg_ctr - test['ctr_prev']
test['low_ctr_flag'] = (test['ctr_gap'] > 0.001).astype(int)
test['quick_win_flag'] = ((test['avg_position_prev'] > 10) & (test['avg_position_prev'] <= 20) &
                           (test['imp_prev15'] >= 50)).astype(int)
baseline_scores = 0.6 * test['low_ctr_flag'] + 0.4 * test['quick_win_flag']

comparison = pd.DataFrame({
    "method": ["Week-4 baseline rule", "Random Forest (this week)"],
    "Precision@50": [
        precision_at_k(baseline_scores.values, y_test.values, 50),
        precision_at_k(model_scores, y_test.values, 50),
    ]
})
print(comparison.to_string(index=False))

                   method  Precision@50
     Week-4 baseline rule          0.24
Random Forest (this week)          0.66


/tmp/ipykernel_2302/2748592165.py:20: FutureWarning: The default of observed=False is deprecated and will be changed to True in a future version of pandas. Pass observed=False to retain current behavior or observed=True to adopt the future default and silence this warning.
  tier_avg_ctr = test.groupby('position_tier')['ctr_prev'].transform('mean')


## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

Error and interpretation:

Permutation importance ranks ctr_prev highest (0.0070), followed by
impression_volatility_prev (0.0023) and avg_position_prev (0.0013).
imp_prev15 contributes essentially nothing (slightly negative, i.e. noise).
All importances are small in absolute terms — consistent with the earlier
finding that no single pre-decision signal separates decliners well on its
own (AUC ~0.5 for each feature individually).

The model is heavily conservative: 0 confident false positives (score > 0.7
predicting decline that didn't happen) but 2,342 confident false negatives
(score < 0.3 for items that actually declined). This means the model rarely
commits to a "declining" prediction at all — it defaults toward predicting
stability, and when it's wrong, it's wrong in the direction of missing real
declines rather than crying wolf.

Looking at the false-negative sample: these are items with real volume
(100–2,500+ impressions) and decent position (2–4), so the model isn't
failing on thin/noisy rows — it's failing on genuinely established pages
that still declined, which the pre-decision features here don't seem to
predict well. This points to a real limitation of the feature set (no
freshness/content-age signal, no query-mix signal) rather than a modeling
mistake.

Takeaway: the model doesn't clearly outperform the baseline on raw
precision if it can't decide confidently, but it does surface exactly
which features carry the most (still weak) signal — useful direction for
capstone feature work, not a finished result.

In [6]:
from sklearn.inspection import permutation_importance

perm = permutation_importance(model, X_test, y_test, n_repeats=10, random_state=42)
importance = pd.DataFrame({
    "feature": feature_cols,
    "importance": perm.importances_mean
}).sort_values("importance", ascending=False)
print(importance.to_string(index=False))

# Confidently wrong: model predicted high probability of decline but item didn't decline (or vice versa)
test_eval = test.copy()
test_eval['model_score'] = model_scores
test_eval['true_label'] = y_test.values
false_positives = test_eval[(test_eval['model_score'] > 0.7) & (test_eval['true_label'] == 0)]
false_negatives = test_eval[(test_eval['model_score'] < 0.3) & (test_eval['true_label'] == 1)]

print(f"\nConfident false positives (predicted decline, didn't decline): {len(false_positives)}")
print(f"Confident false negatives (predicted stable, actually declined): {len(false_negatives)}")
print("\nSample false positives:")
print(false_positives[['imp_prev15', 'avg_position_prev', 'ctr_prev']].head())
print("\nSample false negatives:")
print(false_negatives[['imp_prev15', 'avg_position_prev', 'ctr_prev']].head())

                   feature  importance
                  ctr_prev    0.006970
impression_volatility_prev    0.002331
         avg_position_prev    0.001278
                imp_prev15   -0.000118

Confident false positives (predicted decline, didn't decline): 0
Confident false negatives (predicted stable, actually declined): 2342

Sample false positives:
Empty DataFrame
Columns: [imp_prev15, avg_position_prev, ctr_prev]
Index: []

Sample false negatives:
    imp_prev15  avg_position_prev  ctr_prev
5       2029.0           2.799633  0.008628
36       179.0           4.386034  0.009375
40      1128.0           1.618665  0.003512
64      2477.0           3.929196  0.004456
67       106.0           4.109554  0.006993


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.